In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import tifffile
import dask.array as da
import utils
import skimage as ski
import pandas as pd
# import time
# import extract_features
# import ast
from cellpose.models import CellposeModel
from cellpose.io import logger_setup
logger_setup()
import dask_image.ndfilters as dask_ndi
# from deeptile.extensions import stitch
# import deeptile
import importlib
importlib.reload(utils)
from brieflow_segment_utils import image_log_scale

In [ ]:
img_path = "/Users/hannahbolen/Desktop/image_analysis/slide_tiff/o8p_day24_s12.ome.tif"
coverslip_path = "/Users/hannahbolen/Desktop/image_analysis/masks_and_results/o8p_day24_s12_coverslip.tif"
save_path = "/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/o8p_day24_s12_cytoplasm_mask.tif"
img_dtype = tifffile.TiffFile(img_path).pages[0].dtype
y0 = 16000
x0 = 14000
img = da.from_zarr(tifffile.imread(img_path, aszarr=True))[:,y0:y0+6400, x0:x0+6400]
# coverslip = da.from_zarr(tifffile.imread(coverslip_path, aszarr=True))[y0:y0+6400, x0:x0+6400]

In [ ]:
img_log = image_log_scale(img)

In [ ]:
nuclei_mask = da.from_zarr(tifffile.imread("/Users/hannahbolen/Desktop/image_analysis/masks_and_results/o8p_day24_s12_nuclei_mask.tif", aszarr=True))[y0:y0+6400, x0:x0+6400]

In [ ]:
model_parameters = {'gpu': True}
eval_parameters = {'diameter':100, 'batch_size':16, 'normalize':False}
model = CellposeModel(**model_parameters)
mask, _, _ = model.eval(img_log, **eval_parameters)
# cellpose = utils.cellpose_segmentation(model_parameters, eval_parameters, threshold=0)
# masks_nuclei = cellpose(tiles_nuclei)
# mask_nuclei = stitch.stitch_masks(masks_nuclei)

# tifffile.imwrite(save_path, mask_nuclei.astype(img_dtype))
fig, ax = plt.subplots(figsize=(20,20))
ax.imshow(ski.segmentation.mark_boundaries(img_log[1], nuclei_mask, color=(1,0,1)))
ax.set_yticks([])
ax.set_xticks([])
plt.tight_layout()


In [ ]:
from brieflow_segment_utils import reconcile_nuclei_cells

In [ ]:
nuclei, cells = reconcile_nuclei_cells(nuclei_mask.compute(), mask, how='contained_in_cells')

In [ ]:
tifffile.imwrite("/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/reconcile_cyto_mask.tif", cells.astype(img_dtype))
tifffile.imwrite("/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/reconcile_nuclei_mask.tif", nuclei.astype(img_dtype))

In [ ]:
fig, ax = plt.subplots(ncols=2,figsize=(20,10))
# fig, ax = plt.subplots(figsize=(20,20))

ax[0].imshow(ski.segmentation.mark_boundaries(ski.segmentation.mark_boundaries(img_log[0], nuclei, color=(1,0,0), mode='thick'), cells, color=(1,0,1)), cmap='gray')
ax[1].imshow(ski.segmentation.mark_boundaries(ski.segmentation.mark_boundaries(img_log[1], nuclei, color=(1,0,0), mode='thick'), cells, color=(1,0,1)), cmap='gray')
# # ax[1].imshow(ski.color.label2rgb(mask))
# # ax[1].set_yticks([])
# # ax[1].set_xticks([])
# # plt.tight_layout()


# # # #img2 = da.from_zarr(tifffile.imread(img_path, aszarr=True))[1,y0:y1,y0:y1]

# # cy5 = da.from_zarr(tifffile.imread(img_path, aszarr=True))[1,y0:y1,y0:y1]
# # ax.imshow(ski.segmentation.mark_boundaries(ski.exposure.rescale_intensity(cy5, in_range=(0,4000)), mask, color=(1,0,1), mode='thick'), cmap='gray')
# ax.imshow(tiles_nuclei[10,10], cmap='gray')


# ax[0].imshow(img_log[0], cmap='gray')
# ax[1].imshow(img_log[1], cmap='gray')


# ax.set_yticks([])
# ax.set_xticks([])
ax[0].set_yticks([])
ax[0].set_xticks([])
ax[1].set_yticks([])
ax[1].set_xticks([])

plt.tight_layout()

In [ ]:
# preprocessing
img0 = img[0].copy()
img1 = img[1].copy()

# log transform and normalize GFP channel
img0 = da.log(da.where(img0 == 0, 0.01, img0))
x01 = da.percentile(img0, 0.1, axis=(0, 1)).compute()
x99 = da.percentile(img0, 99, axis=(0, 1)).compute()
img0 = (img0 - x01) / (x99 - x01)

# clip and normalize CY5
p99_9 = da.percentile(img1, 99.9, axis=(0, 1)).compute()
img1 = da.clip(img1, 0, p99_9)/p99_9

# stack and gaussian blur
img = da.stack([img0, img1])
img=dask_ndi.gaussian_filter(img, sigma=(0,2,2))
#img = img*coverslip

In [ ]:
# # preprocessing
# p01 = da.percentile(img[0], 0.1, axis=(0, 1)).compute()
p99 = da.percentile(img[1], 99, axis=(0, 1)).compute()
p99_9 = da.percentile(img[1], 99.9, axis=(0, 1)).compute()

img_clip99_9 = dask_ndi.gaussian_filter(da.clip(img[1], 0, p99_9)/p99_9, sigma=2)

img_log = da.log(da.where(img[0] == 0, 0.01, img[0]))
x01 = da.percentile(img_log, 0.1, axis=(0, 1)).compute()
x99 = da.percentile(img_log, 99, axis=(0, 1)).compute()
img_log = (img_log - x01) / (x99 - x01)
img_log=dask_ndi.gaussian_filter(img_log, sigma=2)
# img = img*coverslip
# print(x01,x99)

In [ ]:
fig, ax = plt.subplots(figsize=(20,20))
ax.imshow(ski.segmentation.mark_boundaries(img[0], nuclei_mask.compute(), color=(1,0,1)))
ax.set_yticks([])
ax.set_xticks([])
plt.tight_layout()

In [ ]:
tifffile.imwrite("/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/cyto_mask.tif", mask.astype(img_dtype))